In [1]:
import os
import copy
import random
import collections
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt

# Scikit-learn Preprocessing, Splitting, and Metrics
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, f1_score, balanced_accuracy_score, matthews_corrcoef, precision_recall_curve, auc, brier_score_loss
from sklearn.calibration import calibration_curve, CalibrationDisplay

# Imbalanced Learning Frameworks
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek

# Machine Learning & Deep Learning Frameworks
from xgboost import XGBClassifier
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Explainable AI (XAI) Libraries
import shap
import lime
import lime.lime_tabular

import preprocess

In [2]:
def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    # Ensure fully deterministic behavior in PyTorch backends
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [3]:
filepath = "data/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv"
df = preprocess.load_and_clean_dataset(filepath)

#change to a binary label
df['Label'] = df['Label'].astype(str).str.strip().str.upper()
df['Label'] = df['Label'].apply(lambda x: 0 if x == 'BENIGN' else 1)

X = df.drop(columns=['Label'])
y = df['Label'].values

print(f"[*] Cleaned Feature matrix shape: {X.shape}")
print(f"[*] Target distribution: Benign (0) = {np.sum(y == 0)}, Attack (1) = {np.sum(y == 1)}")


[*] Loading raw dataset from: data/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv


c:\Users\Moritz\Documents\Uni\bachlor_thesis\ids-xai-thesis\preprocess.py:12: DtypeWarning: Columns (0: Flow ID, 1:  Source IP, 2:  Destination IP, 3:  Timestamp, 4:  Label) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, encoding='latin-1')


[-] Dropped identifier columns: ['Flow ID', 'Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Timestamp']
[!] Purged 288737 malformed rows containing Infinity or empty cells.
[*] Cleaned Feature matrix shape: (170231, 78)
[*] Target distribution: Benign (0) = 168051, Attack (1) = 2180


In [4]:
# Step A: Split off 30% of the data into a temporary block, stratifying to preserve class ratios
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

# Step B: Split the temporary block evenly to yield a 15% Validation set and a 15% Test set
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print("==================== DATA SPLIT SUMMARY ====================")
print(f"Training Set (70%):   X = {X_train.shape}, y = {y_train.shape}")
print(f"Validation Set (15%): X = {X_val.shape}, y = {y_val.shape}")
print(f"Test Set (15%):       X = {X_test.shape}, y = {y_test.shape}")

==================== DATA SPLIT SUMMARY ====================
Training Set (70%):   X = (119161, 78), y = (119161,)
Validation Set (15%): X = (25535, 78), y = (25535,)
Test Set (15%):       X = (25535, 78), y = (25535,)


In [5]:
# 1. Fit scaler ONLY on training data
scaler = MinMaxScaler()
scaler.fit(X_train)

X_train_scaled = pd.DataFrame(scaler.transform(X_train), columns=X_train.columns)
X_val_scaled = pd.DataFrame(scaler.transform(X_val), columns=X_val.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

# 2. Correlated Feature Groups (Supervisor point 15 & 16)
corr_matrix = X_train_scaled.corr(method="spearman")
# Get upper triangle of correlation matrix
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
# Find columns with correlation > 0.90
to_drop = [column for column in upper.columns if any(upper[column].abs() > 0.90)]

print(f"[*] Features identified for potential removal (>0.90 correlation): {len(to_drop)}")

[*] Features identified for potential removal (>0.90 correlation): 39


In [6]:
# Initialize SMOTE-Tomek with a fixed seed for strict reproducibility
smote_tomek = SMOTETomek(random_state=42)

# Resample ONLY the training set
X_train_resampled, y_train_resampled = smote_tomek.fit_resample(
    X_train_scaled,
    y_train
)

print("================= RESAMPLING SUMMARY =================")
print(f"Original Training Class Ratios: 0 = {np.sum(y_train == 0)}, 1 = {np.sum(y_train == 1)}")
print(f"Resampled Training Data Shape:  X = {X_train_resampled.shape}, y = {y_train_resampled.shape}")
print(f"Resampled Training Class Ratios: 0 = {np.sum(y_train_resampled == 0)}, 1 = {np.sum(y_train_resampled == 1)}")

================= RESAMPLING SUMMARY =================
Original Training Class Ratios: 0 = 117635, 1 = 1526
Resampled Training Data Shape:  X = (235254, 78), y = (235254,)
Resampled Training Class Ratios: 0 = 117627, 1 = 117627


In [7]:
print("[*] Initializing XGBoost Classifier...")

# Initialize XGBoost with strict reproducibility parameters
xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=0.1,
    random_state=42,
    use_label_encoder=False,
    early_stopping_rounds=15
)

# Fit model with early stopping monitored against the validation split
xgb_model.fit(
    X_train_resampled, 
    y_train_resampled,
    eval_set=[(X_val_scaled, y_val)],
    verbose=False
)

print(f"[*] XGBoost training complete.")
print(f"    -> Best Iteration: {xgb_model.best_iteration}")

[*] Initializing XGBoost Classifier...


c:\Users\Moritz\Documents\Uni\bachlor_thesis\ids-xai-thesis\thesis_env\Lib\site-packages\xgboost\callback.py:385: UserWarning: [14:38:02] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()


[*] XGBoost training complete.
    -> Best Iteration: 398


In [8]:
class RobustNetworkSecurityDNN(nn.Module):
    def __init__(self, input_dim):
        super(RobustNetworkSecurityDNN, self).__init__()
        
        # Layer 1: Input to Hidden 1
        self.fc1 = nn.Linear(input_dim, 128)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(p=0.30)
        
        # Layer 2: Hidden 1 to Hidden 2
        self.fc2 = nn.Linear(128, 64)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(p=0.30)
        
        # Layer 3: Hidden 2 to Output Layer
        self.fc3 = nn.Linear(64, 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        x = self.dropout1(self.relu1(self.fc1(x)))
        x = self.dropout2(self.relu2(self.fc2(x)))
        x = self.sigmoid(self.fc3(x))
        return x

# Set calculation device backend
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_features_count = X_train_resampled.shape[1]

# Instantiate model architecture
model_dnn = RobustNetworkSecurityDNN(input_dim=input_features_count).to(device)
print(f"[*] PyTorch Network mapped successfully onto target hardware device: {device.type.upper()}")

[*] PyTorch Network mapped successfully onto target hardware device: CPU


In [9]:
# Convert DataFrames/Arrays to PyTorch multi-dimensional tensors
train_dataset = TensorDataset(
    torch.FloatTensor(X_train_resampled.values),
    torch.FloatTensor(y_train_resampled).unsqueeze(1)
)
val_x_tensor = torch.FloatTensor(X_val_scaled.values).to(device)
val_y_tensor = torch.FloatTensor(y_val).unsqueeze(1).to(device)

# Configure data loader iterations
train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)

# Set optimizer equations and standard binary loss scoring
criterion = nn.BCELoss()
optimizer = optim.Adam(model_dnn.parameters(), lr=0.001)

# Early Stopping parameters
patience = 10
best_val_loss = float('inf')
best_model_weights = None
patience_counter = 0
max_epochs = 150

print("[*] Initiating DNN optimization loop...")
for epoch in range(1, max_epochs + 1):
    model_dnn.train()
    running_loss = 0.0
    
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = model_dnn(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * batch_x.size(0)
        
    # Validation Evaluation Phase (Zero Leakage Check)
    model_dnn.eval()
    with torch.no_grad():
        val_outputs = model_dnn(val_x_tensor)
        val_loss = criterion(val_outputs, val_y_tensor).item()
        
    epoch_train_loss = running_loss / len(train_dataset)
    
    # Early stopping criteria tracking
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_weights = copy.deepcopy(model_dnn.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1
        
    if patience_counter >= patience:
        print(f"[*] Early stopping triggered at Epoch {epoch}. Overfitting threshold neutralized.")
        break

# Roll back parameter matrix states to the optimal captured validation validation loss weights
if best_model_weights is not None:
    model_dnn.load_state_dict(best_model_weights)
print(f"[*] Restored optimal model configurations. Best Validation Loss: {best_val_loss:.5f}")

C:\Users\Moritz\AppData\Local\Temp\ipykernel_26428\4040257355.py:6: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  val_x_tensor = torch.FloatTensor(X_val_scaled.values).to(device)


[*] Initiating DNN optimization loop...
[*] Early stopping triggered at Epoch 21. Overfitting threshold neutralized.
[*] Restored optimal model configurations. Best Validation Loss: 0.04361


In [10]:
# Helper function to generate clean inference probability matrices from our DNN
def get_dnn_probabilities(df_input):
    model_dnn.eval()
    with torch.no_grad():
        tensor_input = torch.FloatTensor(df_input.values).to(device)
        probs = model_dnn(tensor_input).cpu().numpy().flatten()
    return probs

# Step A: Collect raw feature inference array probabilities from validation subsets
xgb_val_probs = xgb_model.predict_proba(X_val_scaled)[:, 1]
dnn_val_probs = get_dnn_probabilities(X_val_scaled)

# Step B: Declare target tuning thresholds range
thresholds_pool = np.arange(0.50, 0.96, 0.05)

best_xgb_threshold = 0.50
best_xgb_f1 = 0.0
best_dnn_threshold = 0.50
best_dnn_f1 = 0.0

print("================= VAL THRESHOLD CALIBRATION =================")
for t in thresholds_pool:
    # Evaluate XGBoost arrays
    xgb_preds = (xgb_val_probs >= t).astype(int)
    xgb_f1 = f1_score(y_val, xgb_preds, zero_division=0)
    if xgb_f1 > best_xgb_f1:
        best_xgb_f1 = xgb_f1
        best_xgb_threshold = t
        
    # Evaluate DNN arrays
    dnn_preds = (dnn_val_probs >= t).astype(int)
    dnn_f1 = f1_score(y_val, dnn_preds, zero_division=0)
    if dnn_f1 > best_dnn_f1:
        best_dnn_f1 = dnn_f1
        best_dnn_threshold = t
        
    print(f"Threshold: {t:.2f} | XGB F1: {xgb_f1:.4f} | DNN F1: {dnn_f1:.4f}")

print("\n[*] Calibrated Decision Parameter Options Frozen:")
print(f"    -> Selected Frozen XGBoost Threshold: {best_xgb_threshold:.2f} (Val F1: {best_xgb_f1:.4f})")
print(f"    -> Selected Frozen DNN Threshold:     {best_dnn_threshold:.2f} (Val F1: {best_dnn_f1:.4f})")

================= VAL THRESHOLD CALIBRATION =================
Threshold: 0.50 | XGB F1: 0.9954 | DNN F1: 0.5195
Threshold: 0.55 | XGB F1: 0.9954 | DNN F1: 0.5304
Threshold: 0.60 | XGB F1: 0.9954 | DNN F1: 0.5450
Threshold: 0.65 | XGB F1: 0.9954 | DNN F1: 0.6054
Threshold: 0.70 | XGB F1: 0.9954 | DNN F1: 0.6681
Threshold: 0.75 | XGB F1: 0.9970 | DNN F1: 0.7687
Threshold: 0.80 | XGB F1: 0.9970 | DNN F1: 0.8626
Threshold: 0.85 | XGB F1: 0.9969 | DNN F1: 0.8555
Threshold: 0.90 | XGB F1: 0.9969 | DNN F1: 0.8547
Threshold: 0.95 | XGB F1: 0.9985 | DNN F1: 0.8571

[*] Calibrated Decision Parameter Options Frozen:
    -> Selected Frozen XGBoost Threshold: 0.95 (Val F1: 0.9985)
    -> Selected Frozen DNN Threshold:     0.80 (Val F1: 0.8626)


In [11]:
# Step A: Extract test predictions using frozen configurations
xgb_test_probs = xgb_model.predict_proba(X_test_scaled)[:, 1]
dnn_test_probs = get_dnn_probabilities(X_test_scaled)

xgb_test_preds = (xgb_test_probs >= best_xgb_threshold).astype(int)
dnn_test_preds = (dnn_test_probs >= best_dnn_threshold).astype(int)

# Step B: Print formal validation summaries for your thesis report
print("=================== FINAL FROZEN XGBOOST REPORT ===================")
print(f"Applied Decision Threshold: {best_xgb_threshold:.2f}")
print(confusion_matrix(y_test, xgb_test_preds))
print(classification_report(y_test, xgb_test_preds, digits=4))

print("\n==================== FINAL FROZEN DNN REPORT ====================")
print(f"Applied Decision Threshold: {best_dnn_threshold:.2f}")
print(confusion_matrix(y_test, dnn_test_preds))
print(classification_report(y_test, dnn_test_preds, digits=4))

=================== FINAL FROZEN XGBOOST REPORT ===================
Applied Decision Threshold: 0.95
[[25207     1]
 [    4   323]]
              precision    recall  f1-score   support

           0     0.9998    1.0000    0.9999     25208
           1     0.9969    0.9878    0.9923       327

    accuracy                         0.9998     25535
   macro avg     0.9984    0.9939    0.9961     25535
weighted avg     0.9998    0.9998    0.9998     25535


==================== FINAL FROZEN DNN REPORT ====================
Applied Decision Threshold: 0.80
[[25126    82]
 [   19   308]]
              precision    recall  f1-score   support

           0     0.9992    0.9967    0.9980     25208
           1     0.7897    0.9419    0.8591       327

    accuracy                         0.9960     25535
   macro avg     0.8945    0.9693    0.9286     25535
weighted avg     0.9966    0.9960    0.9962     25535



In [12]:
# Standardized probability wrappers for the explainers
def xgb_predict_proba_wrapper(x_numpy):
    # Map numpy matrix rows directly back to pandas format to preserve column naming indices natively
    df_temp = pd.DataFrame(x_numpy, columns=X_train_scaled.columns)
    return xgb_model.predict_proba(df_temp)

def dnn_predict_proba_wrapper(x_numpy):
    df_temp = pd.DataFrame(x_numpy, columns=X_train_scaled.columns)
    probs_class_1 = get_dnn_probabilities(df_temp)
    probs_class_0 = 1.0 - probs_class_1
    return np.column_stack((probs_class_0, probs_class_1))

In [13]:
# Map feature list and order cleanly from the step 1 variables
feature_list = X_train_scaled.columns.tolist()
x_train_unresampled_numpy = X_train_scaled.values

# Initialize LIME Tabular Explainer with fixed seed for absolute reproducibility
lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=x_train_unresampled_numpy,
    feature_names=feature_list,
    class_names=["Benign", "Web Attack"],
    mode="classification",
    kernel_width=None, # Standard baseline width (to be evaluated later)
    random_state=42
)

In [14]:
# Clear, unresampled background sample for both KernelExplainers
shap_background_baseline = shap.sample(X_train_scaled, 100, random_state=42)

shap_explainer_xgb = shap.KernelExplainer(
    model=xgb_predict_proba_wrapper,
    data=shap_background_baseline
)

shap_explainer_dnn = shap.KernelExplainer(
    model=dnn_predict_proba_wrapper,
    data=shap_background_baseline
)


In [15]:
def extract_aligned_lime_attributions(instance, predict_fn, num_features):
    """
    Generates local linear surrogate attributions and extracts weights directly 
    via feature indices to completely prevent string interval truncation errors.
    """
    # Force explanation extraction explicitly for Class 1 (Web Attack)
    exp = lime_explainer.explain_instance(
        data_row=instance,
        predict_fn=predict_fn,
        labels=(1,),
        num_features=num_features
    )
    
    # Isolate the underlying index-to-weight mapping array for Class 1
    raw_local_exp = exp.local_exp[1]
    index_to_weight = {feature_idx: weight for feature_idx, weight in raw_local_exp}
    
    # Reconstruct the uniform vector following native column positioning
    total_features = x_train_unresampled_numpy.shape[1]
    lime_vector = np.zeros(total_features)
    for idx in range(total_features):
        lime_vector[idx] = index_to_weight.get(idx, 0.0)
        
    # Capture R-squared local surrogate training fidelity
    fidelity_score = exp.score
    
    return lime_vector, fidelity_score


def extract_aligned_shap_attributions(instance, explainer):
    """
    Extracts local SHAP attribution vectors aligned explicitly to Class 1 (Web Attack)
    natively via unified KernelExplainer multi-output tracking.
    """
    # Unified execution path: sampling permutations with a baseline budget of 100
    raw_values = explainer.shap_values(instance, nsamples=100)
    
    # Handle structural differences across different SHAP / NumPy versions dynamically
    if isinstance(raw_values, list):
        # Format A: List of length 2 -> Index 1 isolates Class 1 (Attack)
        shap_vector = raw_values[1]
    elif isinstance(raw_values, np.ndarray):
        # Format B: NumPy Array -> Inspect dimensions to pull Class 1
        if len(raw_values.shape) == 2:
            if raw_values.shape[1] == 2:   # Shape is (78, 2) -> Column 1 is Class 1
                shap_vector = raw_values[:, 1]
            elif raw_values.shape[0] == 2: # Shape is (2, 78) -> Row 1 is Class 1
                shap_vector = raw_values[1, :]
            else:
                shap_vector = raw_values
        else:
            shap_vector = raw_values
    else:
        shap_vector = raw_values
        
    return np.array(shap_vector).flatten()


In [16]:
def tune_lime_hyperparameters(
    X_train_data,
    X_val_data,
    predict_fn,
    feature_names,
    kernel_widths=[0.25, 0.5, 1.0, 2.0,2.5, None], #0.1 results in 1.0 proably because artificial collaps
    num_samples_list=[1000, 3000, 5000, 8000],
    val_sample_size=100
):
    """
    Evaluates LIME hyperparameter combinations on validation data to optimize R^2 fidelity.
    """
    # Sample a representative subset of validation data for hyperparameter selection
    val_subset = X_val_data.sample(n=min(val_sample_size, len(X_val_data)), random_state=42).values
    num_features = X_train_data.shape[1]
    
    results = []

    print("--- Running LIME Hyperparameter Search on Validation Data ---")
    for kw in kernel_widths:
        # Re-initialize explainer for each kernel width setting
        temp_explainer = lime.lime_tabular.LimeTabularExplainer(
            training_data=X_train_data.values,
            feature_names=feature_names,
            class_names=["Benign", "Web Attack"],
            mode="classification",
            kernel_width=kw,
            random_state=42
        )
        
        for ns in num_samples_list:
            scores = []
            for instance in val_subset:
                exp = temp_explainer.explain_instance(
                    data_row=instance,
                    predict_fn=predict_fn,
                    labels=(1,),
                    num_features=num_features,
                    num_samples=ns
                )
                scores.append(exp.score)
            
            mean_r2 = np.mean(scores)
            median_r2 = np.median(scores)
            
            results.append({
                'kernel_width': str(kw),
                'num_samples': ns,
                'mean_r2': mean_r2,
                'median_r2': median_r2,
                'std_r2': np.std(scores)
            })
            
            print(f"Kernel Width: {str(kw):<5} | Samples: {ns:<5} | Mean R2: {mean_r2:.4f} | Median R2: {median_r2:.4f}")

    results_df = pd.DataFrame(results).sort_values(by='mean_r2', ascending=False)
    
    best_params = results_df.iloc[0]
    print("\n================ Optimal Parameters Selected ================")
    print(f"Best Kernel Width: {best_params['kernel_width']}")
    print(f"Best Num Samples : {best_params['num_samples']}")
    print(f"Validation R2    : {best_params['mean_r2']:.4f}")
    print("=============================================================")
    
    return results_df, best_params

# Run parameter search on validation split for XGBoost
xgb_lime_tuning_df, best_xgb_params = tune_lime_hyperparameters(
    X_train_data=X_train_scaled,
    X_val_data=X_val_scaled,
    predict_fn=xgb_predict_proba_wrapper,
    feature_names=feature_list
)

--- Running LIME Hyperparameter Search on Validation Data ---
Kernel Width: 0.25  | Samples: 1000  | Mean R2: 0.0000 | Median R2: 0.0000
Kernel Width: 0.25  | Samples: 3000  | Mean R2: 0.0000 | Median R2: 0.0000
Kernel Width: 0.25  | Samples: 5000  | Mean R2: 0.0000 | Median R2: 0.0000
Kernel Width: 0.25  | Samples: 8000  | Mean R2: 0.0000 | Median R2: 0.0000
Kernel Width: 0.5   | Samples: 1000  | Mean R2: 0.0000 | Median R2: 0.0000
Kernel Width: 0.5   | Samples: 3000  | Mean R2: 0.0000 | Median R2: 0.0000
Kernel Width: 0.5   | Samples: 5000  | Mean R2: 0.0000 | Median R2: 0.0000
Kernel Width: 0.5   | Samples: 8000  | Mean R2: 0.0000 | Median R2: 0.0000
Kernel Width: 1.0   | Samples: 1000  | Mean R2: 0.0001 | Median R2: 0.0000
Kernel Width: 1.0   | Samples: 3000  | Mean R2: 0.0002 | Median R2: 0.0000
Kernel Width: 1.0   | Samples: 5000  | Mean R2: 0.0003 | Median R2: 0.0000
Kernel Width: 1.0   | Samples: 8000  | Mean R2: 0.0004 | Median R2: 0.0000
Kernel Width: 2.0   | Samples: 1000  |

In [17]:
import warnings
from sklearn.exceptions import ConvergenceWarning

def generate_per_observation_evaluations(
    X_eval,
    predict_fn,
    shap_explainer,
    feature_names,
    kernel_width,
    num_samples,
    eval_sample_size=300,
    random_state=42
):
    """
    Runs LIME and SHAP across a representative sample of observations, 
    tracking per-instance fidelity (R2) and vector similarity metrics.
    """
    # 1. Subsample evaluation data to prevent hours of computation
    if len(X_eval) > eval_sample_size:
        print(f"[!] X_eval has {len(X_eval)} rows. Subsampling {eval_sample_size} representative instances for analysis...")
        eval_subset = X_eval.sample(n=eval_sample_size, random_state=random_state).reset_index(drop=True)
    else:
        eval_subset = X_eval.reset_index(drop=True)
        
    # Re-instantiate LIME explainer with selected kernel width
    lime_exp = lime.lime_tabular.LimeTabularExplainer(
        training_data=X_train_scaled.values,
        feature_names=feature_names,
        class_names=["Benign", "Web Attack"],
        mode="classification",
        kernel_width=kernel_width,
        random_state=random_state
    )
    
    records = []
    total_features = eval_subset.shape[1]
    
    print(f"--- Computing Attributions & Fidelity across {len(eval_subset)} observations ---")
    
    # Context manager to suppress scikit-learn / SHAP singular matrix warnings
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=ConvergenceWarning)
        warnings.filterwarnings("ignore", category=UserWarning, module="shap")
        warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")
        
        for idx in range(len(eval_subset)):
            instance = eval_subset.iloc[idx].values
            
            # 1. Extract LIME attribution & per-instance local fidelity R2
            exp = lime_exp.explain_instance(
                data_row=instance,
                predict_fn=predict_fn,
                labels=(1,),
                num_features=total_features,
                num_samples=num_samples
            )
            lime_fid = exp.score  # R2 local surrogate score
            
            raw_local = exp.local_exp[1]
            idx_to_w = {f_idx: w for f_idx, w in raw_local}
            lime_vec = np.array([idx_to_w.get(i, 0.0) for i in range(total_features)])
            
            # 2. Extract SHAP attribution
            shap_vec = extract_aligned_shap_attributions(instance, shap_explainer)
            
            # 3. Compute Similarity / Agreement Metrics
            norm_product = (np.linalg.norm(lime_vec) * np.linalg.norm(shap_vec))
            cosine_sim = (np.dot(lime_vec, shap_vec) / norm_product) if norm_product > 0 else 0.0
            
            spearman_corr, _ = stats.spearmanr(lime_vec, shap_vec)
            if np.isnan(spearman_corr):
                spearman_corr = 0.0
                
            records.append({
                'obs_idx': idx,
                'lime_fidelity_r2': lime_fid,
                'cosine_similarity': cosine_sim,
                'spearman_corr': spearman_corr,
                'lime_vec': lime_vec,
                'shap_vec': shap_vec
            })
            
            # Print progress indicator every 50 samples
            if (idx + 1) % 50 == 0 or (idx + 1) == len(eval_subset):
                print(f"  Progress: {idx + 1}/{len(eval_subset)} observations completed.")
            
    return pd.DataFrame(records)

In [18]:
opt_xgb_kw = best_xgb_params['kernel_width']
opt_xgb_ns = best_xgb_params['num_samples']

# Execute evaluation on a 300-sample subset of validation data
xgb_fidelity_eval_df = generate_per_observation_evaluations(
    X_eval=X_val_scaled,
    predict_fn=xgb_predict_proba_wrapper,
    shap_explainer=shap_explainer_xgb,
    feature_names=feature_list,
    kernel_width=opt_xgb_kw,
    num_samples=opt_xgb_ns,
    eval_sample_size=300
)

[!] X_eval has 25535 rows. Subsampling 300 representative instances for analysis...
--- Computing Attributions & Fidelity across 300 observations ---
  Progress: 50/300 observations completed.
  Progress: 100/300 observations completed.
  Progress: 150/300 observations completed.
  Progress: 200/300 observations completed.
  Progress: 250/300 observations completed.
  Progress: 300/300 observations completed.


In [19]:
def compare_fidelity_stratifications(eval_df, threshold_type='median', custom_threshold=0.3):
    """
    Splits observations into High vs. Low LIME fidelity cohorts 
    and compares LIME vs. SHAP agreement across both groups.
    """
    if threshold_type == 'median':
        split_val = eval_df['lime_fidelity_r2'].median()
        print(f"Stratifying by Median R2 Threshold: {split_val:.4f}\n")
    else:
        split_val = custom_threshold
        print(f"Stratifying by Custom R2 Threshold: {split_val:.4f}\n")
        
    high_fid = eval_df[eval_df['lime_fidelity_r2'] >= split_val]
    low_fid = eval_df[eval_df['lime_fidelity_r2'] < split_val]
    
    summary_data = {
        'Cohort': ['High Fidelity Group', 'Low Fidelity Group'],
        'Count': [len(high_fid), len(low_fid)],
        'Mean LIME R2': [high_fid['lime_fidelity_r2'].mean(), low_fid['lime_fidelity_r2'].mean()],
        'Median LIME R2': [high_fid['lime_fidelity_r2'].median(), low_fid['lime_fidelity_r2'].median()],
        'Mean Cosine Sim': [high_fid['cosine_similarity'].mean(), low_fid['cosine_similarity'].mean()],
        'Mean Spearman Corr': [high_fid['spearman_corr'].mean(), low_fid['spearman_corr'].mean()]
    }
    
    summary_df = pd.DataFrame(summary_data)
    
    print("=================== STRATIFIED AGREEMENT COMPARISON ===================")
    print(summary_df.to_string(index=False))
    print("=======================================================================")
    
    # Hypothesis Verification Check
    diff_cosine = summary_df.loc[0, 'Mean Cosine Sim'] - summary_df.loc[1, 'Mean Cosine Sim']
    diff_spearman = summary_df.loc[0, 'Mean Spearman Corr'] - summary_df.loc[1, 'Mean Spearman Corr']
    
    print("\n[+] Hypothesis Analysis:")
    if diff_cosine > 0 and diff_spearman > 0:
        print("-> Confirmed: High LIME fidelity observations show noticeably stronger agreement with SHAP.")
        print("   This indicates that disagreement in low-fidelity samples is driven by LIME surrogate poor fit.")
    else:
        print("-> Disagreement persists even in high LIME fidelity observations, suggesting fundamental")
        print("   methodological differences (additive global game theory vs. localized ridge surrogate).")

    return summary_df

# Run stratification analysis
stratification_summary = compare_fidelity_stratifications(xgb_fidelity_eval_df, threshold_type='median')

Stratifying by Median R2 Threshold: 0.0796

=================== STRATIFIED AGREEMENT COMPARISON ===================
             Cohort  Count  Mean LIME R2  Median LIME R2  Mean Cosine Sim  Mean Spearman Corr
High Fidelity Group    150      0.138394        0.098240         0.128139            0.077010
 Low Fidelity Group    150      0.061679        0.063367         0.196768            0.121309

[+] Hypothesis Analysis:
-> Disagreement persists even in high LIME fidelity observations, suggesting fundamental
   methodological differences (additive global game theory vs. localized ridge surrogate).
